# PaceAlgo ML — Notebook 03: Asset Clustering

**Zweck:** Datengetriebene Identifizierung von "Asset-Persönlichkeiten" via K-Means. Statt Ticker-Strings zu hardcoden (`BTC→high_vol`) bestimmen wir aus den tatsächlichen Markteigenschaften welche Symbole sich ähnlich verhalten.

**Cluster-Features (7 Aggregate pro Symbol):**
- vol_level (mean atr%)
- vol_volatility (std of atr%)
- vol_realized (mean realized vol)
- trend_strength (mean ADX)
- pct_trending (% Bars mit ADX > 25)
- bb_width_avg
- volume_dispersion

**Method:**
- StandardScaler
- K-Means mit k=2..5 → bestes k via Silhouette-Score
- Cluster werden nach vol_level sortiert (Cluster 0 = niedrigste vol)

**Output:** `artifacts/asset_clusters.parquet` mit (symbol, tf, cluster, cluster_label, ...)

**Laufzeit:** ~30 Sek (liest nur Aggregate, nicht alle 5M Bars).

## 1. Environment Setup

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_PROCESSED = Path('/content/processed')  # written by notebook 02
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_PROCESSED, ARTIFACTS_REPORTS
    ARTIFACTS = ARTIFACTS_REPORTS.parent

ARTIFACTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
print(f'Processed input: {DATA_PROCESSED}')
print(f'Artifacts out:   {ARTIFACTS}')

In [ ]:
!pip install -q scikit-learn matplotlib 2>&1 | tail -1

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from core.features import (
    build_asset_profile_table,
    cluster_assets,
    label_clusters_semantic,
    CLUSTER_FEATURE_BUILDERS,
)
print('Dependencies ready')

## 2. Build Asset Profile Table

Für jedes Feature-File werden 7 aggregierte Eigenschaften berechnet.

In [ ]:
feature_files = sorted(DATA_PROCESSED.glob('*_features.parquet'))
print(f'Found {len(feature_files)} feature files')

profile = build_asset_profile_table(feature_files)
print(f'\nAsset profile table ({profile.shape[0]} rows × {profile.shape[1]} cols):')
display(profile.round(4))

## 3. K-Means Clustering with Silhouette Score Selection

In [ ]:
clustered, info = cluster_assets(profile, k_range=range(2, 6))
clustered = label_clusters_semantic(clustered)

print(f'Chosen k: {info["chosen_k"]}')
print(f'\nSilhouette scores per k:')
for k, s in info['silhouette_per_k'].items():
    marker = ' ← chosen' if k == info['chosen_k'] else ''
    print(f'  k={k}: {s:.4f}{marker}')

print(f'\nCluster assignments:')
display(clustered[['symbol','tf','cluster','cluster_label','vol_level','trend_strength','pct_trending']].round(4))

## 4. Cluster Centers (Personality Profiles)

In [ ]:
centers = info['cluster_centers'].copy()
centers.index = [f'cluster_{i}' for i in range(len(centers))]
centers_sem = label_clusters_semantic(
    pd.DataFrame({'cluster': range(len(centers)), 'vol_level': centers['vol_level'].values})
)
centers.index = centers_sem['cluster_label'].values
print('Cluster centers (un-scaled, original feature space):')
display(centers.round(5))

## 5. Visualization — Vol vs Trend Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, info['chosen_k']))
for cluster_id in sorted(clustered['cluster'].unique()):
    sub = clustered[clustered['cluster'] == cluster_id]
    label = sub['cluster_label'].iloc[0]
    ax.scatter(sub['vol_level'] * 100, sub['trend_strength'],
                s=120, c=[colors[cluster_id]], label=f'{label} (k={cluster_id})',
                edgecolors='black', linewidths=1, alpha=0.85)
    for _, row in sub.iterrows():
        ax.annotate(f"{row['symbol']}/{row['tf']}",
                     (row['vol_level'] * 100, row['trend_strength']),
                     xytext=(7, 5), textcoords='offset points', fontsize=9)
ax.set_xlabel('Volatility level (ATR% — average)')
ax.set_ylabel('Trend strength (ADX — average)')
ax.set_title('PaceAlgo Asset Clustering: Volatility × Trend')
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Save to Artifacts (lives on Drive)

In [ ]:
out_path = ARTIFACTS / 'asset_clusters.parquet'
clustered.to_parquet(out_path, compression='zstd')
print(f'Saved: {out_path}')
print(f'Size: {out_path.stat().st_size / 1024:.1f} KB')

centers_path = ARTIFACTS / 'cluster_centers.parquet'
centers.to_parquet(centers_path, compression='zstd')
print(f'Saved: {centers_path}')

# Save cluster info for Pine export later
import json
info_serializable = {
    'chosen_k': info['chosen_k'],
    'silhouette_per_k': {str(k): float(v) for k, v in info['silhouette_per_k'].items()},
    'feature_cols': info['feature_cols'],
    'scaler_mean': info['scaler_mean'],
    'scaler_scale': info['scaler_scale'],
}
info_path = ARTIFACTS / 'cluster_info.json'
with open(info_path, 'w') as f:
    json.dump(info_serializable, f, indent=2)
print(f'Saved: {info_path}')

---

## Nächster Schritt

→ `04_triple_barrier_labeling.ipynb`

Triple-Barrier-Labels (Marcos López de Prado) — pro Bar: trifft TP zuerst (+1), SL zuerst (-1), oder Time-Barrier (0)? Wir testen R ∈ {1.5, 2.0, 2.5} und nutzen die Cluster-Zuordnung für optimales R-Multiplier pro Asset-Klasse.